In [1]:
# ============================================================
# CATHEDRAL ONE-AXIOM FREEZE v1.0
#
# Axiom:
#   Unphased chaos, forced to exhaust entropy without external scale,
#   closes into the minimal self-dual finite shell.
#
# This script derives:
#   D, N, V, E, F, q
#   icosahedral shell graph
#   shell and centred spectra
#   spectral phi
#   gamma, d_star, d_cl, Delta
#   hidden 8 = 4 + 4 face-kernel exhaust
#   W_scalar = 9/5
#   chi_signed = 2/sqrt5
#   alpha inverse root
#   electron gravitational coupling alpha_G_e
#
# No observed constants.
# No target ledger.
# No measured alpha.
# No measured G.
# No inserted phi coordinates.
# No inserted face list.
# ============================================================

import numpy as np
import math
import itertools
import json
import hashlib

print("=" * 78)
print("CATHEDRAL ONE-AXIOM FREEZE v1.0")
print("=" * 78)

pi = math.pi
ln2 = math.log(2.0)

def laplacian(A):
    return np.diag(A.sum(1)) - A

def add_edge(A, i, j):
    A[i, j] = 1.0
    A[j, i] = 1.0

# ------------------------------------------------------------
# GATE 1
# Dimension from exterior self-duality:
# dim Lambda^1 = dim Lambda^2
# D = D(D-1)/2
# ------------------------------------------------------------

D = None
for d in range(2, 50):
    if d == d * (d - 1) // 2:
        D = d
        break

if D is None:
    raise RuntimeError("No nontrivial exterior self-dual dimension found.")

# ------------------------------------------------------------
# GATE 2
# Entropy shell:
# choose integer N nearest to Shannon closure
# (N - D - 1) / N ~= ln 2
# continuous crossing: N = (D+1)/(1-ln2)
# ------------------------------------------------------------

N_cont = float(D + 1) / (1.0 - ln2)
N = int(round(N_cont))
V = N - 1

entropy_dim = N - D - 1
gamma = 1.0 / float(entropy_dim * entropy_dim)
entropy_ratio = float(entropy_dim) / float(N)
T_geo = ln2 - entropy_ratio

# ------------------------------------------------------------
# GATE 3
# Closed triangular shell:
# V - E + F = 2
# 3F = 2E
# therefore E = 3V - 6, F = 2V - 4
# q = 2E/V
# ------------------------------------------------------------

E = 3 * V - 6
F = 2 * V - 4
q_float = float(2 * E) / float(V)

if abs(q_float - round(q_float)) > 1.0e-14:
    raise RuntimeError("Valence is not integral.")

q = int(round(q_float))

if V != 2 * q + 2:
    raise RuntimeError("The entropy shell does not close into a two-pole q-ring shell.")

# ------------------------------------------------------------
# GATE 4
# Canonical representative of the unique q=5 regular triangular shell.
#
# This does not insert coordinates or faces.
# It builds the labelled graph from:
#   two poles
#   two q-cycles
#   nearest diagonal closure
#
# For the derived q=5 this is the icosahedral graph.
# ------------------------------------------------------------

top = 0
bottom = V - 1

ring_a = list(range(1, 1 + q))
ring_b = list(range(1 + q, 1 + 2 * q))

A_shell = np.zeros((V, V))

for a in ring_a:
    add_edge(A_shell, top, a)

for b in ring_b:
    add_edge(A_shell, bottom, b)

for i in range(q):
    add_edge(A_shell, ring_a[i], ring_a[(i + 1) % q])
    add_edge(A_shell, ring_b[i], ring_b[(i + 1) % q])

    add_edge(A_shell, ring_a[i], ring_b[i])
    add_edge(A_shell, ring_a[i], ring_b[(i - 1) % q])

E_check = int(A_shell.sum() / 2.0)
deg = A_shell.sum(1)

if E_check != E:
    raise RuntimeError("Graph edge count failed.")

if int(np.min(deg)) != q or int(np.max(deg)) != q:
    raise RuntimeError("Graph is not regular with derived valence.")

# Derive triangular faces from the graph, not from a face list.
faces = []

for tri in itertools.combinations(range(V), 3):
    a, b, c = tri
    if A_shell[a, b] > 0.5 and A_shell[a, c] > 0.5 and A_shell[b, c] > 0.5:
        faces.append(tri)

F_check = len(faces)

if F_check != F:
    raise RuntimeError("Face count failed.")

# ------------------------------------------------------------
# GATE 5
# Spectra and spectral phi extraction
# ------------------------------------------------------------

L_shell = laplacian(A_shell)
ev_shell, U_shell = np.linalg.eigh(L_shell)
vals_shell, cnt_shell = np.unique(np.round(ev_shell, 12), return_counts=True)

A13 = np.zeros((N, N))
A13[1:N, 1:N] = A_shell
A13[0, 1:N] = 1.0
A13[1:N, 0] = 1.0

L13 = laplacian(A13)
ev13 = np.linalg.eigvalsh(L13)
vals13, cnt13 = np.unique(np.round(ev13, 12), return_counts=True)

triplet_vals = []

for i in range(len(vals_shell)):
    if int(cnt_shell[i]) == 3:
        triplet_vals.append(float(vals_shell[i]))

triplet_vals = sorted(triplet_vals)

if len(triplet_vals) != 2:
    raise RuntimeError("Could not extract two triplet frequencies.")

sqrt5_spectral = (triplet_vals[1] - triplet_vals[0]) / 2.0
phi = (1.0 + sqrt5_spectral) / 2.0

# ------------------------------------------------------------
# GATE 6
# Entropy rail and bandwidth
# ------------------------------------------------------------

d_star = (1.0 - gamma) * pi / (float(N) * phi)
d_cl = float(D) / float(F)
Delta = d_cl - d_star

# ------------------------------------------------------------
# GATE 7
# Hidden face kernel
# ------------------------------------------------------------

B = np.zeros((F, V))

for i, tri in enumerate(faces):
    for v in tri:
        B[i, v] = 1.0

rankB = int(np.linalg.matrix_rank(B, tol=1.0e-10))
hidden_total_dim = F - rankB

A_face = np.zeros((F, F))

for i in range(F):
    for j in range(i + 1, F):
        shared = len(set(faces[i]) & set(faces[j]))
        if shared == 2:
            A_face[i, j] = 1.0
            A_face[j, i] = 1.0

L_face = laplacian(A_face)

Utmp, Stmp, Vhtmp = np.linalg.svd(B.T, full_matrices=True)
rankBT = int(np.linalg.matrix_rank(B.T, tol=1.0e-10))

Q_hidden_raw = Vhtmp.T[:, rankBT:]

L_hidden = Q_hidden_raw.T.dot(L_face).dot(Q_hidden_raw)
hidden_evals, hidden_vecs = np.linalg.eigh(L_hidden)
Q_hidden = Q_hidden_raw.dot(hidden_vecs)

hidden_vals, hidden_cnt = np.unique(np.round(hidden_evals, 12), return_counts=True)

hidden_groups = []

for val in hidden_vals:
    Qg = Q_hidden[:, np.abs(hidden_evals - float(val)) < 1.0e-8]
    hidden_groups.append((float(val), Qg))

# ------------------------------------------------------------
# Visible projectors
# ------------------------------------------------------------

projectors = []

for val in vals_shell:
    cols = []

    for k in range(len(ev_shell)):
        if abs(ev_shell[k] - float(val)) < 1.0e-8:
            cols.append(k)

    Qp = U_shell[:, cols]
    P = Qp.dot(Qp.T)
    projectors.append((float(val), len(cols), P))

P_scalar = projectors[0][2]
P_triplet_low = projectors[1][2]
P_five = projectors[2][2]
P_triplet_high = projectors[3][2]

def sym_return(Q):
    cols = []

    for i in range(Q.shape[1]):
        for j in range(i, Q.shape[1]):
            h = Q[:, i] * Q[:, j]

            if j > i:
                h = math.sqrt(2.0) * h

            cols.append(B.T.dot(h))

    return np.column_stack(cols)

def mixed_return(QA, QB):
    cols = []

    for i in range(QA.shape[1]):
        for j in range(QB.shape[1]):
            h = QA[:, i] * QB[:, j]
            cols.append(B.T.dot(h))

    return np.column_stack(cols)

def projector_power(P, M):
    X = P.dot(M)
    v = np.linalg.norm(X, "fro")
    return float(v * v)

def signature(label, M):
    p_scalar = projector_power(P_scalar, M)
    p_low = projector_power(P_triplet_low, M)
    p_five = projector_power(P_five, M)
    p_high = projector_power(P_triplet_high, M)

    p_triplet = p_low + p_high

    W = 0.0
    if p_five > 1.0e-14:
        W = p_scalar / p_five

    chi = 0.0
    if p_triplet > 1.0e-14:
        chi = (p_low - p_high) / p_triplet

    return {
        "label": label,
        "scalar": float(p_scalar),
        "triplet_low": float(p_low),
        "five": float(p_five),
        "triplet_high": float(p_high),
        "scalar_five_ratio": float(W),
        "chirality": float(chi),
        "rank": int(np.linalg.matrix_rank(M, tol=1.0e-10))
    }

returns = []

for val, Qg in hidden_groups:
    returns.append(signature("sym_hidden_%0.12g" % val, sym_return(Qg)))

for i in range(len(hidden_groups)):
    for j in range(i + 1, len(hidden_groups)):
        va, Qa = hidden_groups[i]
        vb, Qb = hidden_groups[j]
        returns.append(signature("mixed_hidden_%0.12g_%0.12g" % (va, vb), mixed_return(Qa, Qb)))

returns.append(signature("sym_hidden_all", sym_return(Q_hidden)))

# Canonical scalar channel:
# choose even scalar-five return with dominant fivefold power.
even_candidates = []

for item in returns:
    trip = abs(item["triplet_low"]) + abs(item["triplet_high"])

    if item["scalar"] > 1.0e-12 and item["five"] > 1.0e-12 and trip < 1.0e-12:
        even_candidates.append(item)

if len(even_candidates) == 0:
    raise RuntimeError("No scalar-five even channel found.")

scalar_channel = sorted(even_candidates, key=lambda z: (-z["five"], z["label"]))[0]
W_scalar = scalar_channel["scalar_five_ratio"]

# Canonical chiral channel:
# choose pure triplet mixed return with maximal absolute chirality.
chiral_candidates = []

for item in returns:
    sf = abs(item["scalar"]) + abs(item["five"])
    trip = abs(item["triplet_low"]) + abs(item["triplet_high"])

    if trip > 1.0e-12 and sf < 1.0e-12:
        chiral_candidates.append(item)

if len(chiral_candidates) == 0:
    raise RuntimeError("No pure chiral triplet channel found.")

chiral_channel = sorted(chiral_candidates, key=lambda z: (-abs(z["chirality"]), z["label"]))[0]
chi_signed = chiral_channel["chirality"]

# ------------------------------------------------------------
# GATE 8
# Root scalar charge
# ------------------------------------------------------------

charge_integer_capacity = N * N - E - (D - 1)

charge_gap_correction = Delta * (
    float(N) + W_scalar - (1.0 + gamma) / float(D)
)

charge_entropy_dressing = -Delta * Delta * float(entropy_dim) / float(q * N)

alpha_inv_root = (
    float(charge_integer_capacity)
    + charge_gap_correction
    + charge_entropy_dressing
)

alpha_root = 1.0 / alpha_inv_root

# ------------------------------------------------------------
# GATE 9
# Gravity coupling
#
# Dimensionless electron gravitational coupling:
#
# alpha_G_e = alpha^21 * A_G
#
# 21 = 13 + 8
# 13 = centred carrier
# 8 = hidden exhaust
# ------------------------------------------------------------

depth = N + hidden_total_dim

A_G = (
    float(D + 1) / float(D)
    * (d_star / d_cl)
    * (1.0 - Delta / (d_cl * (float(N - 2) - gamma)))
)

alpha_G_e = math.pow(alpha_root, depth) * A_G
m_Pl_over_m_e = 1.0 / math.sqrt(alpha_G_e)

# ------------------------------------------------------------
# Ordered invariant ledger
# ------------------------------------------------------------

hidden_branch_dim = float(hidden_groups[0][1].shape[1])

ledger = []

def emit(label, value):
    ledger.append([label, float(value)])

emit("root_scalar_charge_inverse", alpha_inv_root)
emit("root_scalar_charge", alpha_root)
emit("weak_singlet_ratio_D_over_N", float(D) / float(N))
emit("strong_face_ratio_F_over_N2", float(F) / float(N * N))
emit("first_face_screen_current", 1.0 / math.sqrt(float(F) - 1.0 / hidden_branch_dim))
emit("first_chiral_gap_current", Delta * (math.pow(phi, 6.0) - 1.0))
emit("direct_spatial_gap_current", float(D) * Delta / float(D - 1))
emit("first_cp_area", gamma * Delta * (1.0 + W_scalar * gamma))
emit("solar_triplet_residue", 1.0 / float(D) - float(V) * Delta)
emit("atmospheric_triplet_residue", 0.5 + float(F) * Delta + gamma)
emit("reactor_entropy_pair", 2.0 * gamma * (d_star / d_cl) * (d_star / d_cl) * (1.0 - float(F) * Delta))
emit("leptonic_gap_area", -float(N) * Delta)
emit("matter_closure", float(2 * D) / float(F - 1))
emit("vacuum_closure", float(N) / float(F - 1))
emit("visible_matter_fraction", (float(D - 1) / float(N)) * (float(2 * D) / float(F - 1)))
emit("hidden_matter_fraction", ((float(N) - float(D - 1)) / float(N)) * (float(2 * D) / float(F - 1)))
emit("scalar_tilt_residue", 1.0 - float(D) * gamma + Delta)
emit("tensor_entropy_gap", 16.0 * gamma * Delta)
emit("running_gap_square", -Delta * Delta)
emit("neutrino_ladder_seed", d_star + d_star * d_star)
emit("top_gap_complement", 1.0 - Delta)
emit("electron_gravity_coupling", alpha_G_e)
emit("planck_to_electron_ratio", m_Pl_over_m_e)

# ------------------------------------------------------------
# Freeze hash
# ------------------------------------------------------------

freeze = {
    "axiom": "unphased chaos forced by entropy exhaustion into the minimal self-dual finite shell",
    "D": int(D),
    "V": int(V),
    "N": int(N),
    "E": int(E),
    "F": int(F),
    "q": int(q),
    "entropy_dim": int(entropy_dim),
    "gamma": float(gamma),
    "T_geo": float(T_geo),
    "shell_spectrum_values": [float(x) for x in vals_shell],
    "shell_spectrum_counts": [int(x) for x in cnt_shell],
    "centered_spectrum_values": [float(x) for x in vals13],
    "centered_spectrum_counts": [int(x) for x in cnt13],
    "sqrt5_spectral": float(sqrt5_spectral),
    "phi": float(phi),
    "d_star": float(d_star),
    "d_cl": float(d_cl),
    "Delta": float(Delta),
    "hidden_total_dim": int(hidden_total_dim),
    "hidden_values": [float(x) for x in hidden_vals],
    "hidden_counts": [int(x) for x in hidden_cnt],
    "W_scalar": float(W_scalar),
    "chi_signed": float(chi_signed),
    "alpha_inv_root": float(alpha_inv_root),
    "alpha_root": float(alpha_root),
    "gravity_depth": int(depth),
    "A_G": float(A_G),
    "alpha_G_e": float(alpha_G_e),
    "m_Pl_over_m_e": float(m_Pl_over_m_e),
    "ledger": ledger
}

freeze_text = json.dumps(freeze, sort_keys=True, separators=(",", ":"))
freeze_hash = hashlib.sha256(freeze_text.encode("utf-8")).hexdigest()

# ------------------------------------------------------------
# Print freeze
# ------------------------------------------------------------

print()
print("AXIOM")
print("  unphased chaos -> entropy exhaustion -> minimal self-dual finite shell")

print()
print("GATES")
print("  D                         =", D)
print("  N_cont                    = %.15f" % N_cont)
print("  V                         =", V)
print("  N                         =", N)
print("  entropy_dim                =", entropy_dim)
print("  gamma                      = %.15f" % gamma)
print("  T_geo                      = %.15f" % T_geo)
print("  E                         =", E)
print("  F                         =", F)
print("  q                         =", q)

print()
print("SHELL SPECTRUM")
for i in range(len(vals_shell)):
    print("  %.12f multiplicity %d" % (float(vals_shell[i]), int(cnt_shell[i])))

print()
print("CENTERED SPECTRUM")
for i in range(len(vals13)):
    print("  %.12f multiplicity %d" % (float(vals13[i]), int(cnt13[i])))

print()
print("SPECTRAL CONSTANTS")
print("  sqrt5_spectral             = %.15f" % sqrt5_spectral)
print("  phi                        = %.15f" % phi)

print()
print("ENTROPY RAIL")
print("  d_star                     = %.15f" % d_star)
print("  d_cl                       = %.15f" % d_cl)
print("  Delta                      = %.15f" % Delta)

print()
print("HIDDEN FACE-KERNEL")
print("  rankB                      =", rankB)
print("  hidden_total_dim            =", hidden_total_dim)
print("  hidden spectrum             =", [float(x) for x in hidden_vals], [int(x) for x in hidden_cnt])
print("  scalar_channel              =", scalar_channel["label"])
print("  W_scalar                   = %.15f" % W_scalar)
print("  chiral_channel              =", chiral_channel["label"])
print("  chi_signed                 = %.15f" % chi_signed)

print()
print("ROOT SCALAR CHARGE")
print("  integer capacity            = %.15f" % float(charge_integer_capacity))
print("  gap correction              = %.15f" % charge_gap_correction)
print("  entropy dressing            = %.15e" % charge_entropy_dressing)
print("  alpha_inv_root              = %.15f" % alpha_inv_root)
print("  alpha_root                  = %.15e" % alpha_root)

print()
print("GRAVITY")
print("  depth N + hidden_dim        =", depth)
print("  A_G                         = %.15f" % A_G)
print("  alpha_G_e                   = %.15e" % alpha_G_e)
print("  m_Pl / m_e                  = %.15e" % m_Pl_over_m_e)

print()
print("ORDERED INVARIANT LEDGER")
for label, value in ledger:
    if abs(value) < 1.0e-4 and value != 0.0:
        print("  %-32s %.15e" % (label, value))
    else:
        print("  %-32s %.15f" % (label, value))

print()
print("FREEZE HASH")
print("  sha256")
print("    " + freeze_hash)

print()
print("STATUS")
print("  observed constants used     = 0")
print("  target ledger used          = 0")
print("  measured alpha used         = 0")
print("  measured gravity used       = 0")
print("  phi hardcoded               = no")
print("  coordinates hardcoded       = no")
print("  faces hardcoded             = no")
print("  root alpha emitted          = yes")
print("  gravity emitted             = yes")

CATHEDRAL ONE-AXIOM FREEZE v1.0

AXIOM
  unphased chaos -> entropy exhaustion -> minimal self-dual finite shell

GATES
  D                         = 3
  N_cont                    = 13.035565413083717
  V                         = 12
  N                         = 13
  entropy_dim                = 9
  gamma                      = 0.012345679012346
  T_geo                      = 0.000839488252253
  E                         = 30
  F                         = 20
  q                         = 5

SHELL SPECTRUM
  -0.000000000000 multiplicity 1
  2.763932022500 multiplicity 3
  6.000000000000 multiplicity 5
  7.236067977500 multiplicity 3

CENTERED SPECTRUM
  -0.000000000000 multiplicity 1
  3.763932022500 multiplicity 3
  7.000000000000 multiplicity 5
  8.236067977500 multiplicity 3
  13.000000000000 multiplicity 1

SPECTRAL CONSTANTS
  sqrt5_spectral             = 2.236067977500000
  phi                        = 1.618033988750000

ENTROPY RAIL
  d_star                     = 0.14751081015957